# ETL — Tabla Central de Hechos (`fct_video_metricas`)

Este notebook extrae los videos únicos desde la capa Silver (`tiktok_data_eng.silver.silver_tiktok`),
calcula el identificador único `row_hash` (SHA-256 de `video_id | video_url`),
resuelve las claves foráneas hacia las dimensiones (`dim_autor` y `dim_fecha`),
reúne las métricas cuantitativas puras (`plays`, `likes`, `comments`, `shares`, `saves`, `duration`),
calcula la tasa `engagement_rate` y ejecuta un `MERGE` idempotente en `tiktok_data_eng.gold.fct_video_metricas`.

In [0]:
%sql
-- Inserción / Actualización idempotente (MERGE) en fct_video_metricas utilizando CTEs

WITH videos_deduplicados AS (
    SELECT 
        video_id,
        video_url,
        author_id,
        created_at,
        plays,
        likes,
        comments,
        shares,
        saves,
        duration
    FROM (
        SELECT 
            video_id,
            video_url,
            author_id,
            created_at,
            plays,
            likes,
            comments,
            shares,
            saves,
            duration,
            ROW_NUMBER() OVER (PARTITION BY video_id ORDER BY fecha_ingesta DESC) AS rn
        FROM tiktok_data_eng.silver.silver_tiktok
        WHERE video_id IS NOT NULL
    )
    WHERE rn = 1
),
hechos_con_claves AS (
    SELECT 
        sha2(concat(v.video_id, '|', coalesce(v.video_url, '')), 256) AS row_hash,
        v.video_id,
        a.autor_id,
        f.fecha_id,
        v.plays,
        v.likes,
        v.comments,
        v.shares,
        v.saves,
        v.duration,
        ROUND((v.likes + v.comments + v.shares + v.saves) / NULLIF(v.plays, 0), 4) AS engagement_rate
    FROM videos_deduplicados v
    INNER JOIN tiktok_data_eng.gold.dim_autor a 
        ON v.author_id = a.author_tiktok_id
    INNER JOIN tiktok_data_eng.gold.dim_fecha f 
        ON TO_DATE(v.created_at) = f.fecha
)
MERGE INTO tiktok_data_eng.gold.fct_video_metricas AS target
USING hechos_con_claves AS source
ON target.row_hash = source.row_hash
WHEN MATCHED AND (
    target.plays <=> source.plays = FALSE OR
    target.likes <=> source.likes = FALSE OR
    target.comments <=> source.comments = FALSE OR
    target.shares <=> source.shares = FALSE OR
    target.saves <=> source.saves = FALSE OR
    target.duration <=> source.duration = FALSE OR
    target.engagement_rate <=> source.engagement_rate = FALSE
) THEN UPDATE SET
    target.plays = source.plays,
    target.likes = source.likes,
    target.comments = source.comments,
    target.shares = source.shares,
    target.saves = source.saves,
    target.duration = source.duration,
    target.engagement_rate = source.engagement_rate
WHEN NOT MATCHED THEN INSERT (
    row_hash,
    video_id,
    autor_id,
    fecha_id,
    plays,
    likes,
    comments,
    shares,
    saves,
    duration,
    engagement_rate,
    _created_at
) VALUES (
    source.row_hash,
    source.video_id,
    source.autor_id,
    source.fecha_id,
    source.plays,
    source.likes,
    source.comments,
    source.shares,
    source.saves,
    source.duration,
    source.engagement_rate,
    CURRENT_TIMESTAMP()
);

In [0]:
%sql
-- Validación de calidad, volumetría e integridad referencial en fct_video_metricas
SELECT 
    COUNT(*) AS total_filas,
    COUNT(DISTINCT row_hash) AS total_hashes_unicos,
    COUNT(DISTINCT video_id) AS total_videos_unicos,
    SUM(CASE WHEN row_hash IS NULL THEN 1 ELSE 0 END) AS nulos_pk,
    SUM(CASE WHEN autor_id IS NULL THEN 1 ELSE 0 END) AS huerfanos_autor,
    SUM(CASE WHEN fecha_id IS NULL THEN 1 ELSE 0 END) AS huerfanos_fecha,
    ROUND(AVG(engagement_rate), 4) AS avg_engagement,
    SUM(plays) AS total_plays,
    SUM(likes) AS total_likes
FROM tiktok_data_eng.gold.fct_video_metricas;